In [ ]:
# ======================================================
# Notebook: 6D Hyperparameter Optimisation (Hyperparameter tuning)
# Inputs: (30,6) | Output: (30,)
# Goal: maximise performance
# ======================================================

import numpy as np
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor

# Load data
X = np.load("/mnt/data/initial_inputs.npy")      # (30,6)
y = np.load("/mnt/data/initial_outputs.npy")     # (30,)

# Hyperparameter tuning surrogate
param_dist = {
    "n_estimators": [50,100,200,300],
    "max_depth": [None,3,5,8,12],
    "min_samples_split": [2,5,10],
    "min_samples_leaf": [1,2,4]
}

search = RandomizedSearchCV(
    RandomForestRegressor(),
    param_distributions=param_dist,
    n_iter=25,
    cv=3,
    random_state=42
)

search.fit(X, y)
model = search.best_estimator_

print("Best params:", search.best_params_)

# Candidate sampling
bounds = [(X[:,i].min(), X[:,i].max()) for i in range(6)]
n_candidates = 5000
X_grid = np.column_stack([
    np.random.uniform(b[0], b[1], n_candidates) for b in bounds
])

# Ensemble predictions
preds = np.array([tree.predict(X_grid) for tree in model.estimators_])
mean_pred = preds.mean(axis=0)
uncertainty = preds.std(axis=0)

# Acquisition
acquisition = mean_pred + 0.5 * uncertainty

# Select next (10,6)
top_idx = np.argsort(acquisition)[-10:]
next_points = X_grid[top_idx]

print("Next (10,6) hyperparameter candidates:")
print(next_points)